In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
import pygame
from IPython.display import Audio
from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path
from gymnasium.wrappers import TimeLimit

NUM_ENV = 8
SAVE_DIR="./model-nsm"
TENSORBOARD="./tensorboard-nsm"
TOTAL_TIMESTEP_NUMB = 500_000_000
CHECK_FREQ_NUMB = 5_000
SAVE_FREQ = CHECK_FREQ_NUMB
MAX_STEPS= 12_000

ENT_COEF = 0.001
n_steps=4096
batch_size=64 * NUM_ENV

SAVE_DIR = Path(SAVE_DIR)
combos = [
    [],
    ["LEFT"],
    ["RIGHT"],

    # jump
    ["LEFT", "B"],
    ["RIGHT", "B"],
    # run
    ["LEFT", "X"],
    ["RIGHT", "X"],
    #shake
    ["R2"],
    ["LEFT", "R2"],
    ["RIGHT", "R2"],
]


def make_env(env_id):
    def _init():
        env = make(
            "NewSuperMarioBros-Wii", 
            env_id=env_id,
            #render_mode="human"
        )
        env.set_buttons(["B", "Y", "SELECT", "START", "LEFT", "RIGHT", "DOWN", "UP","A", "X", "L1", "R1", "L2", "R2", "L3", "R3"])

        env = MainDiscretizer(
            env,
            combos,
        )

        env = WarpFrame(env, width=96, height=96)
        # env = WarpFrame(env)
        env = MaxAndSkipEnv(env, skip=4)
        env = TimeLimit(env, MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)
envs = [make_env(i) for i in range(NUM_ENV)]
env = SubprocVecEnv(envs)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

if latest_model_path:
    print(f"Loading existent model: {latest_model_path}")
    model = PPO.load(
    # model = RecurrentPPO.load(
        str(latest_model_path), 
        env=env, 
        verbose=0, 
        tensorboard_log=TENSORBOARD, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
    )
    
else:
    print("None finded, starting from zero.")
    model = PPO('CnnPolicy', 
    # model = RecurrentPPO('CnnLstmPolicy',
        env, 
        verbose=0, 
        # policy_kwargs=policy_kwargs, 
        ent_coef =ENT_COEF,
        n_steps=n_steps,
        batch_size=batch_size,
        tensorboard_log=TENSORBOARD, 
    )

# eval_callback = EvalCallback(
#     eval_env, 
#     best_model_save_path="./logs/best_model",
#     log_path="./logs/results", 
#     eval_freq=5_000,
#     n_eval_episodes=6,
#     deterministic=True
# )

checkpoint_callback=TrainAndLoggingCallback(check_freq=CHECK_FREQ_NUMB, save_path=SAVE_DIR, save_freq=SAVE_FREQ, model=model)
#callback = CallbackList([checkpoint_callback, eval_callback])
callback = CallbackList([checkpoint_callback])

model.learn(total_timesteps=TOTAL_TIMESTEP_NUMB, reset_num_timesteps=False, callback=callback)
model.save("final_nsm")

env.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Loading existent model: model-nsm\best_model_5000
Done Rewards Step Cnt: 90
Done Rewards Step Cnt: 110
Done Rewards Step Cnt: 86
Done Rewards Step Cnt: 100
Done Rewards Step Cnt: 350
Done Rewards Step Cnt: 602
Done Rewards Step Cnt: 109
Done Rewards Step Cnt: 119
Done Rewards Step Cnt: 89
